In [15]:
from pathlib import Path

def find_project_root(start: Path | None = None, marker: str = "requirements.txt") -> Path:
    start = (start or Path.cwd()).resolve()
    for directory in (start, *start.parents):
        if (directory / marker).exists():
            return directory
    raise FileNotFoundError(f"Could not find project root from {start}")

ROOT = find_project_root()
RAW = ROOT / "data" / "raw"

In [33]:
import pandas as pd

dico = pd.read_csv(RAW / "HomeCredit_columns_description.csv", encoding="latin-1", index_col=0)

dico_train = dico[dico["Table"] == "application_{train|test}.csv"]
print(dico_train.shape)
dico_train.head()

(122, 4)


,Table,Row,Description,Special
1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN


In [32]:
#load the data
df_raw = pd.read_csv(RAW / "application_train.csv")

print(df_raw.shape)
df_raw.head()

(307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
#consistence test
csv_columns = set(df_raw.columns)
csv_train = set(dico_train["Row"])

print(csv_train-csv_columns)
print(csv_columns-csv_train)

set()
set()


### Data dictionnary consistency check

**Method:** compared the set of columns names in `application_train.csv` with the `Row` values of the data dictionnary filtered on the application table (set difference in both directions)

**Observation:** both differences are empty - every column in the csv is documented, and every documented variabes exist in the CSV

**Implication:** the dictionnary can be used as the reference for column meanings in the rest of the analysis

In [40]:
print(df_raw.dtypes.value_counts())
print(df_raw.memory_usage(deep=True).sum())
df_raw.info(memory_usage="deep")

#check the version
print(pd.__version__)

float64    65
int64      41
str        16
Name: count, dtype: int64
562761933
<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), str(16)
memory usage: 536.7 MB
3.0.6


### First look recap

**Method:** the `dtypes.value_counts()` and `memory_usage(deep=True)` method to check the consistency and the number of each types

**Observation:** 537 MB and some integer are float64

**Implication:** The data can be used